<a href="https://colab.research.google.com/github/ishikabeniwal/CSET343/blob/main/Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)

In [ ]:
file_name = "heart_failure_clinical_records_dataset.csv"
df = pd.read_csv(file_name)

# Keep an untouched copy for before/after comparison
original_df = df.copy()

print("Dataset loaded successfully.")
print("Shape:", df.shape)

In [ ]:
print("First 5 rows:")
display(df.head())

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDataset Information:")
df.info()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Missing Values Heatmap")
plt.xlabel("Features")
plt.ylabel("Rows")
plt.show()

In [ ]:
numerical_columns = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_creatinine",
    "serum_sodium",
    "time"
]

for column in numerical_columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[column], kde=True)
    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
# Numerical features: median imputation
for column in numerical_columns:
    if df[column].isnull().sum() > 0:
        df[column] = df[column].fillna(df[column].median())

# Categorical/binary features: mode imputation
categorical_columns = [
    "anaemia",
    "diabetes",
    "high_blood_pressure",
    "sex",
    "smoking"
]

for column in categorical_columns:
    if df[column].isnull().sum() > 0:
        df[column] = df[column].fillna(df[column].mode()[0])

print("Missing values after imputation:")
print(df.isnull().sum())

In [ ]:
# Manual IQR outlier detection.
# Outliers are replaced with NaN and then imputed using the median.

for column in numerical_columns:
    values = df[column].dropna().tolist()
    values.sort()

    n = len(values)

    # Manual Q1 and Q3 calculation
    q1_index = (n - 1) // 4
    q3_index = (3 * (n - 1)) // 4

    Q1 = values[q1_index]
    Q3 = values[q3_index]

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    print("\nColumn:", column)
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower Limit:", lower_limit)
    print("Upper Limit:", upper_limit)

    # Replace outliers with NaN
    for index in df.index:
        value = df.loc[index, column]

        if pd.notna(value):
            if value < lower_limit or value > upper_limit:
                df.loc[index, column] = np.nan

# Fill values created by outlier removal
for column in numerical_columns:
    df[column] = df[column].fillna(df[column].median())

print("\nOutlier removal completed.")

In [ ]:
# Check for invalid negative values in features that cannot be negative.
non_negative_columns = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_creatinine",
    "serum_sodium",
    "time"
]

for column in non_negative_columns:
    print("Negative values in", column, ":", (df[column] < 0).sum())

# Replace invalid negative values with the median
for column in non_negative_columns:
    median_value = df[column].median()
    df.loc[df[column] < 0, column] = median_value

print("\nNegative-value correction completed.")

In [ ]:
# age_group: <40, 40-60, >60
df["age_group"] = pd.cut(
    df["age"],
    bins=[-np.inf, 40, 60, np.inf],
    labels=["<40", "40-60", ">60"],
    right=False
)

print(df[["age", "age_group"]].head())

In [ ]:
# risk_score = normalized product of ejection_fraction and serum_creatinine
risk_product = df["ejection_fraction"] * df["serum_creatinine"]

#normalization
minimum = risk_product.min()
maximum = risk_product.max()

if maximum != minimum:
    df["risk_score"] = (risk_product - minimum) / (maximum - minimum)
else:
    df["risk_score"] = 0

print(df[["ejection_fraction", "serum_creatinine", "risk_score"]].head())

In [ ]:
# age_group is categorical, so encode it using one-hot encoding.
# The other listed categorical/binary columns in this dataset are already
# represented as 0/1 values.

df = pd.get_dummies(
    df,
    columns=["age_group"],
    drop_first=True,
    dtype=int
)

print("Data after encoding:")
display(df.head())

In [ ]:
columns_to_scale = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_creatinine",
    "serum_sodium",
    "time"
]

scaler = MinMaxScaler()

df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

print("Data after Min-Max normalization:")
display(df.head())

In [ ]:
print("BEFORE CLEANING")

print("\nMean:")
print(original_df[numerical_columns].mean())

print("\nMedian:")
print(original_df[numerical_columns].median())

print("\nStandard Deviation:")
print(original_df[numerical_columns].std())

In [ ]:
print("AFTER CLEANING AND NORMALIZATION")

print("\nMean:")
print(df[columns_to_scale].mean())

print("\nMedian:")
print(df[columns_to_scale].median())

print("\nStandard Deviation:")
print(df[columns_to_scale].std())

In [ ]:
for column in columns_to_scale:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[column], kde=True)
    plt.title(f"{column} After Cleaning and Normalization")
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
print("Missing values after cleaning:")
print(df.isnull().sum())

total_missing = df.isnull().sum().sum()

print("\nTotal missing values:", total_missing)

if total_missing == 0:
    print("Validation successful: No missing values remain.")
else:
    print("Some missing values remain and should be investigated.")